# 00 — Data pipeline check

Load a raw CSV through the loader + adapter, show the validation report, build the POE panel, and prove the same interface works on a Yahoo-format file.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from sp500rl.config import load_config
from sp500rl.seed import set_seed

CFG = load_config(ROOT / "configs" / "default.yaml")
SEED = set_seed(int(CFG["seed"]))
print(f"seed={SEED}")
print("train", CFG["dates"]["train_start"], "→", CFG["dates"]["train_end"])
print("test ", CFG["dates"]["test_start"], "→", CFG["dates"]["test_end"])


## Canonical CSV (synthetic if needed)

In [ ]:
from sp500rl.data.loaders import load_csv
from sp500rl.data.pipeline import build_panel_from_file
from sp500rl.data.schema import validate
from sp500rl.data.synthetic import write_synthetic
from sp500rl.data.panel import assert_balanced_panel

raw_dir = ROOT / CFG["paths"]["raw_dir"]
raw_dir.mkdir(parents=True, exist_ok=True)
canonical_path = raw_dir / "synthetic_canonical.csv"
if not canonical_path.exists():
    write_synthetic(raw_dir, start=CFG["dates"]["start"], end=CFG["dates"]["end"], seed=SEED)
    print("wrote synthetic CSVs under", raw_dir)

prices = load_csv(canonical_path, adapter="generic")
report = validate(prices)
print(report.summary())
print("tickers", sorted(prices["tic"].unique()))
print("date range", prices["date"].min().date(), "→", prices["date"].max().date())
print("shape", prices.shape)


## Panel + price plot

In [ ]:
import matplotlib.pyplot as plt

panel = build_panel_from_file(canonical_path, cfg=CFG, adapter="generic", universe="sandbox")
assert_balanced_panel(panel, feature_cols=CFG["poe"]["features"])
print("panel shape", panel.shape)
print("panel dates", panel["date"].min().date(), "→", panel["date"].max().date())
print("tickers", sorted(panel["tic"].unique()))

pivot = panel.pivot(index="date", columns="tic", values="close")
ax = pivot.iloc[:, :4].plot(figsize=(10, 4), title="Synthetic close (first 4 tickers)")
ax.set_ylabel("close")
plt.tight_layout()
plt.show()

processed = ROOT / CFG["paths"]["processed_dir"]
processed.mkdir(parents=True, exist_ok=True)
out = processed / "panel_sandbox.parquet"
panel.to_parquet(out, index=False)
print("wrote", out)


## Same interface, Yahoo-format CSV

In [ ]:
yahoo_path = raw_dir / "synthetic_yahoo.csv"
yahoo_prices = load_csv(yahoo_path, adapter="yahoo")
print(validate(yahoo_prices).summary())
yahoo_panel = build_panel_from_file(yahoo_path, cfg=CFG, adapter="yahoo", universe="sandbox")
assert_balanced_panel(yahoo_panel, feature_cols=CFG["poe"]["features"])
print("yahoo panel shape", yahoo_panel.shape, "tickers", sorted(yahoo_panel["tic"].unique()))
assert set(yahoo_panel["tic"].unique()) == set(panel["tic"].unique())
print("Yahoo-format path matches canonical ticker set.")
